# Introdução à Programação(BIA)
## Igor Jesus da Silva
### Matrícula: 202603053
#### E-mail institucional: igorjesus@discente.ufb.br

In [ ]:
!pip install pandas
!pip install numpy
import numpy as np
import pandas as pd

In [ ]:
df_colera = pd.read_csv("/content/saneago_colera_2025.csv")
display(df_colera.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_cd               209 non-null    object 
 1   cd_pai              206 non-null    object 
 2   regiao              207 non-null    object 
 3   populacao_atendida  207 non-null    float64
 4   casos_suspeitos     204 non-null    object 
 5   data_coleta         209 non-null    object 
dtypes: float64(1), object(5)
memory usage: 9.9+ KB


None

In [ ]:
df_analise = df_colera

#======TRATAMENTO DE DADOS======
df_analise['regiao'] = df_analise['regiao'].str.upper()#padronização do texto
df_analise = df_analise.sort_values('cd_pai')# ordena os valores usando como referência a coluna cd_pai
df_analise['casos_suspeitos'] = pd.to_numeric(df_analise['casos_suspeitos'], errors='coerce')#converte a coluna para float e, o que não pode ser convertido, em vazio

df_analise[['casos_suspeitos', 'populacao_atendida']] = df_analise[['casos_suspeitos','populacao_atendida']].where(df_analise[['casos_suspeitos','populacao_atendida']] >= 0, np.nan)#tartamento de números negativos
df_analise[['casos_suspeitos', 'populacao_atendida']] = df_analise[['casos_suspeitos','populacao_atendida']].where(df_analise['casos_suspeitos'] < df_analise['populacao_atendida'], np.nan)#exclui colunas que suspeitos > população
df_analise['data_coleta'] = df_analise['data_coleta'].str.replace(r'[^0-9\-]', '', regex=True)
df_analise['regiao'] = df_analise['regiao'].str.replace(r'^a-zA-Z0-9', '', regex=True)#exclui caracteres especiais

df_analise['regiao'] = df_analise['regiao'].replace('SETOR', '')# "Setor" é algo genérico, então subistitui por vazio
df_analise['regiao'] = df_analise['regiao'].replace('Regiao Desconhecida', '')
df_analise['regiao'] = df_analise['regiao'].replace('', np.nan)# vazio absoluto


df_analise = df_analise[~df_analise['regiao'].str.contains(r'\?', na=False)]
df_analise = df_analise.dropna()# exclui linhas vazias

#======CLASSIFICAÇÃO======
df_analise['taxa_de_incidencia_local'] = df_analise['casos_suspeitos']/df_analise['populacao_atendida']#calcula a incidência local
df_analise['percentual_incidencia'] = taxa_de_incidencia_local * 100 #calcula o percentual de incidência local

#cria função que classifica o nível de alerta,usando média, valor máximo e valor mínimo da coluna "percentual de incidência"
def indicador(risco):
  if risco <= df_analise['percentual_incidencia'].max() * 0.2:
    return 'normal'
  elif df_analise['percentual_incidencia'].max() * 0.2 < risco < df_analise['percentual_incidencia'].mean():
    return 'atenção'
  elif df_analise['percentual_incidencia'].mean() < risco < df_analise['percentual_incidencia'].max() * 0.8:
    return 'Alerta'
  elif df_analise['percentual_incidencia'].max() * 0.8 < risco:
    return 'crítico'
  else:
    return 'ERRO'

#aplica a função "indicador" na coluna "percentual_incidência" e classifica o nível de alerta, gerando a coluna "nível de alerta"
df_analise['nível de alerta'] = df_analise['percentual_incidencia'].apply(indicador)

#======RELATÓRIO======
#produza um relatório com groupby apresentando, por região, a taxa média de
#incidência, o total de casos, o número de CDs e a distribuição de classificações de alerta

df_relatorio = df_analise
#agrupa os dados por região e calcula média de incidência e soma dos casos para cada grupo
df_relatorio = df_relatorio.groupby('regiao').agg(media_incidencia = ('taxa_de_incidencia_local','mean'),
                                                  casos_total = ('casos_suspeitos', 'sum'))

df_alerta = df_analise[['regiao', 'nível de alerta']].drop_duplicates()#cria um df apenas com região e alerta, excluindo duplicações
df_relatorio = df_relatorio.merge(df_alerta,on='regiao',how='left')## faz a junção entre o df_relatorio e o df_alerta
df_relatorio =  df_relatorio.merge(df_analise[['regiao', 'cd_pai']],on='regiao', how='left').drop_duplicates()#combinar ou adicionar "regiao" e "cd_pai" na "cópia".agg
#df_relatorio = df_relatorio.merge(df_analise[['regiao','nível de alerta']], on ='regiao', how='left')

display(df_relatorio)



,regiao,media_incidencia,casos_total,nível de alerta,cd_pai
0,BAIRRO BARRO PRETO NORTE,0.0001388889,10.0000000000,normal,CD013
4,BAIRRO CENTRO APARECIDA,0.0031428571,528.0000000000,Alerta,CD009
8,BAIRRO CENTRO SEN. CANEDO,0.0003515625,45.0000000000,normal,CD014
12,BAIRRO CENTRO TRINDADE,0.0001942857,34.0000000000,normal,CD012
17,BAIRRO CIDADE LIVRE LESTE,0.0035500000,639.0000000000,Alerta,CD010
21,BAIRRO CIDADE LIVRE OESTE,0.0002187500,70.0000000000,normal,CD010
29,BAIRRO GUANABARA I,0.0001153846,21.0000000000,normal,CD008
36,BAIRRO GUANABARA II,0.0001450000,29.0000000000,normal,CD008
44,BAIRRO JARDIM ELDORADO,0.0001868132,34.0000000000,normal,CD014
51,BAIRRO OLIVEIRAS LESTE,0.0001363636,9.0000000000,normal,CD015
